## **化学式だけから無機材料を金属・非金属に分類する**
----

たとえば、TiO₂という化学式のみからこの物質が金属かそうでないかを判定してみる。
化学式から特徴量を求め、その特徴量からRandom Forestを用いて金属・非金属への分類を行う。
データセットとしてMatbenchを用いる。

In [ ]:
# Matminerのインストール
%pip install -q matminer
# ライブラリーの読み込み
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matminer.datasets import load_dataset
from matminer.featurizers.composition import ElementProperty
from pymatgen.core import Composition
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

Matbenchの matbench_expt_is_metal には、4,921件の化学式と金属・非金属のラベルが入っている。
正解ラベルは is_metal で、`1` が金属、`0` が非金属。
このデータに付属するラベルをそのまま使う。

In [ ]:
# データセットの読み込み
df = load_dataset("matbench_expt_is_metal")
# 正解（金属1、非金属0）をyとする
y = df["is_metal"].astype(int)
print(y.value_counts().rename(index={0: "非金属", 1: "金属"}))

`matminer` のMagpieで平均電気陰性度、元素の原子番号、原子半径、価電子数などの特徴量を求める。

TiO₂を例にすると、原子数の割合はTiが \(1/3\)、Oが \(2/3\) 。Tiの原子番号は22、Oは8なので、原子番号だけでも次のような特徴量ができる。

| 特徴量 | TiO₂での値 |
|---|---:|
| 原子数割合で重み付けした平均原子番号 | (22+2*8)/3 ≒ 12.67 |
| 最小原子番号 | 8 |
| 最大原子番号 | 22 |
| 原子番号の範囲 | 22-8=14 |

電気陰性度についても同様に計算する。平均は組成比を反映し、範囲は含まれる元素間の差を表す。 たとえばTiO₂とTi₂O₃では、平均原子番号は異なるが、原子番号の範囲は同じになる。
Magpieが化学式を数値に変換し、Random Forestがその数値から金属・非金属を判定する。

In [ ]:
# 化学式部分(composition)を取り出す
compositions = df["composition"].map(Composition)
# Magpieで特徴量に変換し、ランダムフォレストへの入力Xとする
magpie = ElementProperty.from_preset("magpie", impute_nan=True)
X = pd.DataFrame(
    magpie.featurize_many(compositions.tolist(), pbar=False),
    columns=magpie.feature_labels(),
    index=df.index,
)

print(f"特徴量の数: {X.shape[1]}")

In [ ]:
# 特徴量の中身の表示
print(X)

In [ ]:
# データセットを学習用80% (train), テスト用20% (test)にランダムに分割
train, test = train_test_split(df.index, test_size=0.2, random_state=42, stratify=y)
# 結果を格納するデータフレームを用意し、"actual"に正解データを入れる
results = df.loc[test, ["composition"]].copy()
results["actual"] = y.loc[test]

# ランダムフォレストの学習と予測

In [ ]:
# Random forestの
RF = RandomForestClassifier(n_estimators=200, random_state=42)

# 学習用データで学習を実行
RF.fit(X.loc[train], y.loc[train])

# テスト用データで予測し、結果を"RF"に入れる
results["RF"] = model.predict(X.loc[test])

# 予測精度を計算
print(classification_report(
    results["actual"], results["RF"],
    labels=[0, 1], target_names=["Nonmetal", "Metal"],
    zero_division=0,
    ))

In [ ]:
# 混同行列の計算と表示
ConfusionMatrixDisplay.from_predictions(
    results["actual"], results["RF"],
    labels=[0, 1], display_labels=["Nonmetal", "Metal"],
    cmap="Blues", colorbar=False,
)
plt.show()

In [ ]:
# 予測がうまくいかなかった例を表示
errors = results.loc[
    results["actual"] != results["RF"],
    ["composition", "actual", "RF"],
]
display(errors.head(15))

# 重要特徴量のプロット

金属・非金属を分類する上で重要度が高い特徴量を調べる。

特徴量ごとの重要度をプロットする。

In [ ]:
# RFの結果から重要度を取得
fi = RF.feature_importances_
# 重要度をプロット
plt.plot(fi)
plt.show()

もっとも重要度が高い特徴量を調べる。

In [ ]:
import numpy as np
idx = np.argmax(fi)
print(X.columns[idx])

mean NpValence は、各元素の価電子のうち *p* 軌道に入っている電子の数を、組成中の原子数割合で重み付けした平均。Magpieでは、元素ごとの NpValence を表から取得し、化学式の原子数比を使って平均する。

たとえば TiO₂ では、Tiの価電子配置は \(3d^2 4s^2\) なので NpValence = 0、Oの価電子配置は \(2s^2 2p^4\) なので NpValence = 4 です。Ti 1個とO 2個の割合で平均すると、
\[
\mathrm{mean\ NpValence}=\frac{1\times 0+2\times 4}{1+2}
=\frac{8}{3}\approx 2.67
\]
となる。この 2.67は、構成元素の価電子配置をもとにした組成の記述子。TiO₂結晶内での電荷移動や、結合後の \(p\) 軌道の占有数を計算する工程ではない。